# Data Preparation — Training Manifest with Named Entities

In [1]:
!pip install -U spacy transformers torch torchvision jiwer pandas tqdm torchcodec
!python -m spacy download en_core_web_trf   # or de_dep_news_trf fr_dep_news_trf es_dep_news_trf

Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple
/opt/conda/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 142.3 MB/s  0:00:0200:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


## Load the Kaldi Dataset

In [ ]:
import torchaudio
from IPython.display import Audio, display


# all datasets provide train_nodev (train), train_dev (dev) and test splits
dataset_paths = [
    ("/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en", "voxpopuli"),
    ("/training-1/asr/kaldi_data/cv19/en", "commonvoice"),
    ("/training-1/asr/kaldi_data/singleword/en", "commonvoice_singlewords"),
    ("/training-1/asr/kaldi_data/mls/en", "librispeech"),
                ]# other languages *_de *_fr *_es

def read_split(dataset_path):
    """Read one Kaldi split directory (wav.scp + text) into a sample list."""
    samples = []
    wav_scp_dict = {}
    with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
        for line in rf:
            id, wav_path = line.strip().split("\t")
            wav_scp_dict[id] = wav_path

    text_dict = {}
    with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
        for line in rf:
            id, text = line.strip().split("\t")
            text_dict[id] = text

    for id, wav_path in wav_scp_dict.items():
        text = text_dict[id]
        samples.append({"audio" : wav_path, "text" : text})
    return samples

dataset = []       # train split (train_nodev)
dev_dataset = []   # dev split (train_dev)

for dataset_root, dataset_title in dataset_paths:
    dataset.extend(read_split(f"{dataset_root}/train_nodev"))
    dev_dataset.extend(read_split(f"{dataset_root}/train_dev"))

print(len(dataset), "train samples,", len(dev_dataset), "dev samples")

#helper function: load audio and make sure it is at 16kHz
def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform

first_entry = dataset[0]
print(first_entry)
print("-"*10)
print(first_entry["text"])
audio = load_audio(first_entry["audio"])
display(Audio(audio, rate=16000))

## Named Entity Extraction

In [4]:
import spacy

nlp = spacy.load("en_core_web_trf")   # or de_dep_news_trf fr_dep_news_trf es_dep_news_trf

def extract_entities(text):
    return [e.text for e in nlp(text).ents]

def build_prompt(text):
    seen = []
    for e in extract_entities(text):
        if e not in seen:
            seen.append(e)
    if not seen:
        return None
    return "Vocabulary: " + ", ".join(seen) + "."

print(extract_entities(first_entry["text"]))
print(build_prompt(first_entry["text"]))


/opt/conda/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


['article twenty one']
Vocabulary: article twenty one.


## Write the Training and Dev Manifests

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
import random
MANIFEST_DIR = Path("asr_finetune")   # <-- adapt
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_TRAIN = MANIFEST_DIR / "manifest_train.jsonl"
MANIFEST_DEV = MANIFEST_DIR / "manifest_dev.jsonl"

train_size = 10000
dev_size = 1000

def write_manifest(samples, path, size):
    counter = 0
    random.shuffle(samples)
    with open(path, "w", encoding="UTF-8") as wf:
        for sample in tqdm(samples):
            prompt = build_prompt(sample["text"])
            if prompt is None:
                continue
            entry = {"audio": sample["audio"], "text": sample["text"],
                     "prompt": prompt}
            wf.write(json.dumps(entry, ensure_ascii=False) + "\n")
            counter += 1
            if counter >= size:
                break
    return counter

train_written = write_manifest(dataset, MANIFEST_TRAIN, train_size)
dev_written = write_manifest(dev_dataset, MANIFEST_DEV, dev_size)

print(f"Wrote {train_written} entries to {MANIFEST_TRAIN}")
print(f"Wrote {dev_written} entries to {MANIFEST_DEV}")


  9%|▉         | 166/1753 [00:39<06:13,  4.25it/s]

Wrote 1000 entries to asr_finetune/manifest_train.jsonl
Wrote 100 entries to asr_finetune/manifest_dev.jsonl


## Inspect the Manifests

In [6]:
for path in [MANIFEST_TRAIN, MANIFEST_DEV]:
    print("=" * 70)
    print(path)
    with open(path, encoding="UTF-8") as rf:
        for i, line in enumerate(rf):
            if i >= 5:
                break
            print(json.loads(line))

asr_finetune/manifest_train.jsonl
{'audio': '/training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131007-0900-PLENARY-19-en_20131007-21:26:04_7.wav', 'text': 'this could be done by looking more specifically into the suggested fallback clause in article twenty one in cases brought by the employee against the employer defining as relevant the place of business from which the employee receives or received day to day instructions', 'prompt': 'Vocabulary: article twenty one.'}
{'audio': '/training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131007-0900-PLENARY-19-en_20131007-21:26:04_3.wav', 'text': 'two thousand and twelve for instance the new brussels i regulation provides for the right for employees to sue several employers together and the right for employees to have access to courts in europe even if the employer is domiciled outside europe the commission will', 'prompt': 'Vocabulary: two thousand and twelve, brussels i regulation, europe.'}
{'audio': '/training-1/asr/voxpopuli/